In [10]:
import pandas as pd
import glob

file_paths = glob.glob("*.xlsx")

cleaned_dfs = []

for f in file_paths:
    print("\nProcessing:", f)
    
    df = pd.read_excel(f)
    
    # Clean column names
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.replace(" ", "_")
        .str.replace(".", "", regex=False)
        .str.lower()
    )
    
    # Detect date column
    date_cols = [col for col in df.columns if "date" in col]
    if not date_cols:
        print("⚠️ No date column found — skipping.")
        continue
    
    df["date"] = pd.to_datetime(df[date_cols[0]], errors="coerce")
    df = df[df["date"].notna()]
    
    if len(df) == 0:
        continue
    
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month

    # Remove blank workshop rows
    if "name_of_school" in df.columns:
        df = df[df["name_of_school"].notna()]
    
    if "no_participants" in df.columns:
        df = df[df["no_participants"].notna()]

    # Province extraction (more robust)
    province_cols = ["leinster", "munster", "connacht", "ulster"]
    province_cols_existing = [col for col in province_cols if col in df.columns]

    if province_cols_existing:
        def get_province(row):
            for col in province_cols_existing:
                value = row[col]
                if pd.notna(value) and str(value).strip() not in ["0", "0.0", "nan", ""]:
                    return col.capitalize()
            return None
        
        df["province"] = df.apply(get_province, axis=1)
        df = df.drop(columns=province_cols_existing)
    else:
        df["province"] = None  # 2021 file fallback

    # Identify core columns
    core_cols = [
        "date", "year", "month",
        "name_of_school", "location",
        "no_sessions", "no_participants",
        "province"
    ]

    core_cols_existing = [col for col in core_cols if col in df.columns]

    nationality_cols = [col for col in df.columns if col not in core_cols_existing]

    # Convert nationality columns safely
    df[nationality_cols] = df[nationality_cols].apply(
        pd.to_numeric, errors="coerce"
    )
    df[nationality_cols] = df[nationality_cols].fillna(0)

    df["diversity_count"] = (df[nationality_cols] > 0).sum(axis=1)

    df = df.drop(columns=nationality_cols)

    cleaned_dfs.append(df)

# Merge
master_df = pd.concat(cleaned_dfs, ignore_index=True)

print("\nFinal dataset shape:", master_df.shape)
print("\nProvince distribution:")
print(master_df["province"].value_counts(dropna=False))
print("\nYear distribution:")
print(master_df["year"].value_counts())
print("\nDiversity summary:")
print(master_df["diversity_count"].describe())

master_df.head()






Processing: AD workshop April-June 2023.xlsx
⚠️ No date column found — skipping.

Processing: AD Workshop Aug - Dec 2023.xlsx


C:\Users\lijta\AppData\Local\Temp\ipykernel_24988\270801537.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["diversity_count"] = (df[nationality_cols] > 0).sum(axis=1)



Processing: AD Workshop Sep 25-May26.xlsx


C:\Users\lijta\AppData\Local\Temp\ipykernel_24988\270801537.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["diversity_count"] = (df[nationality_cols] > 0).sum(axis=1)



Processing: ADW Jan - June 2024 .xlsx


C:\Users\lijta\AppData\Local\Temp\ipykernel_24988\270801537.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["diversity_count"] = (df[nationality_cols] > 0).sum(axis=1)



Processing: ADW Jan - May 2025.xlsx


C:\Users\lijta\AppData\Local\Temp\ipykernel_24988\270801537.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["diversity_count"] = (df[nationality_cols] > 0).sum(axis=1)



Processing: ADW Sept - Dec 2024 .xlsx

Processing: Copy of WORKSHOP 2021 Data-9 (1).xlsx

Final dataset shape: (226, 9)

Province distribution:
province
Leinster    141
None         54
Ulster       13
Munster      13
Connacht      5
Name: count, dtype: int64

Year distribution:
year
2025    80
2023    53
2021    49
2024    44
Name: count, dtype: int64

Diversity summary:
count    226.000000
mean      17.747788
std       11.434961
min        2.000000
25%        5.000000
50%       21.000000
75%       27.000000
max       39.000000
Name: diversity_count, dtype: float64
Final cleaned dataset shape: (172, 9)

Province distribution:
province
Leinster    141
Ulster       13
Munster      13
Connacht      5
Name: count, dtype: int64

Year distribution:
year
2025    76
2023    53
2024    43
Name: count, dtype: int64


C:\Users\lijta\AppData\Local\Temp\ipykernel_24988\270801537.py:80: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["diversity_count"] = (df[nationality_cols] > 0).sum(axis=1)
C:\Users\lijta\AppData\Local\Temp\ipykernel_24988\270801537.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df[date_cols[0]], errors="coerce")


In [11]:
# Focus only on 2023–2025
master_df = master_df[master_df["year"].isin([2023, 2024, 2025])]

# Remove rows without province
master_df = master_df[master_df["province"].notna()]

print("Final cleaned dataset shape:", master_df.shape)
print("\nProvince distribution:")
print(master_df["province"].value_counts())
print("\nYear distribution:")
print(master_df["year"].value_counts())

Final cleaned dataset shape: (172, 9)

Province distribution:
province
Leinster    141
Ulster       13
Munster      13
Connacht      5
Name: count, dtype: int64

Year distribution:
year
2025    76
2023    53
2024    43
Name: count, dtype: int64


In [12]:
# Workshops per province
workshops_by_province = (
    master_df.groupby("province")
    .size()
    .reset_index(name="workshops")
)

# Participants per province
participants_by_province = (
    master_df.groupby("province")["no_participants"]
    .sum()
    .reset_index()
)

# Average diversity per province
diversity_by_province = (
    master_df.groupby("province")["diversity_count"]
    .mean()
    .reset_index()
)

print(workshops_by_province)
print(participants_by_province)
print(diversity_by_province)


   province  workshops
0  Connacht          5
1  Leinster        141
2   Munster         13
3    Ulster         13
   province  no_participants
0  Connacht            308.0
1  Leinster           8506.0
2   Munster            752.0
3    Ulster            729.0
   province  diversity_count
0  Connacht        26.800000
1  Leinster        21.184397
2   Munster        26.000000
3    Ulster        22.538462


In [13]:
line_df["workshops_per_100k"] = line_df["workshops_per_100k"].round(2)


NameError: name 'line_df' is not defined